## 1、SummarizationMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization

API：https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware

作用：对历史消息列表进行摘要&总结，达到上下文压缩的效果。

原理：在达到触发条件时，调用大模型对历史消息进行摘要，将摘要的结果作为HumanMessage，放到消息列表最开始的位置。

### 1.1 参数说明

- model：用于摘要的模型，可以是模型名称，也可以是模型对象，如果传递的是模型名称，底层会调用init_chat_model初始化模型。
- trigger：摘要触发条件。是一个列表，每个元素对应一个条件，当任一条件满足时，触发摘要。
    - tokens：token的数量，历史token的累积数量达到该值时触发摘要。
    - messages：历史消息数量，历史消息条数达到该值时触发摘要。
    - fraction：上下文长度比例，历史token的累积数量达到模型的max_input_tokens * fraction时触发摘要。

    如果条件包含fraction，要求模型的profile包含max_input_tokens，Deepseek模型的profile为空，此时需要手动添加该配置项。Deepseek-V3.2的上下文长度为128K。

- keep： 摘要时保留的历史消息，支持三种条件，但是和trigger不同，keep同一时间只能接收一种条件。
    - tokens：摘要时保留的token数量。
    - messages：摘要时保留的历史消息条数。
    - fraction：摘要时保留max_input_tokens * fraction个token。

- token_counter：统计token数量的函数。默认使用LangChain提供的count_tokens_approximately，一般不用更改。对于纯文本消息，该函数的大致思路是先统计消息的字符数，也就是 len(字符串) ，然后再除以每
个token大致的字符数，转换为粗略的token数，再加一些额外开销。作为估算的token数。
- summary_prompt：摘要时的自定义提示词。该提示词需要包含{messages}占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。
- trim_token_to_summarize：摘要时历史消息的最大token数。如果历史消息token数大于该值，则会被裁减。默认为4000。如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。

### 1.2 举例1

测试trigger、keep参数

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # profile=custom_profile,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

agent = create_agent(
    model="deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2)  # 保留最新两条消息
        )
    ]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({
    "messages": messages
})

#print(response)
for msg in response['messages']:
    msg.pretty_print()

### 1.3 举例2

测试summary_prompt参数

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # profile=custom_profile,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

agent = create_agent(
    model="deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2),  # 保留最新两条消息
            summary_prompt="对话历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({
    "messages": messages
})

#print(response)
for msg in response['messages']:
    msg.pretty_print()

## 2、HumanInTheLoopMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop

API：https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HumanInTheLoopMiddleware

HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在 工具调用前 中断Agent运行，等待用户对工具调用请求决策。可选的决策有： approve（同意执行） 、 edit（编辑调用配置后执行） 、 reject（拒绝执行） 。

### 2.1 参数说明

- interrupt_on：工具名和中断策略的映射。策略可以是True、False或InterruptOnConfig对象，精细控制决策选项。比如：
```
interrupt_on={
    "get_weather": True,
    "read_email_tool": False,
    "send_email_tool": {
        "allowed_decisions": ["approve", "reject"],
    },
}
```
True表示所有决策(approve, edit, reject) 都可以选择，相当于
```
"get_weather": {
    "allowed_decisions": ["approve", "edit", "reject"],
}
```
False表示不中断，即无需审批即可执行。

InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
① allowed_decisions 精细控制中断后允许的决策。
② description ：特定工具的中断描述信息，优先级高于description_prefix，后 者会更改 所有 工具中断的描述。

- description_prefix：自定义中断描述，默认为"Tool execution requires approval"。

### 2.2 举例1:调用前中断

注意：本例需要从中断的位置让Agent继续运行，这就需要用到短期记忆，这里先使用即可。

创建Agent时通过checkpointer参数启用了短期记忆，在调用时通过传递相同的config加载记忆。

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from rich import print as rprint

from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断"
                }
            },
            description_prefix="中断了"
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气，"
            "查询今日新闻，"
            "查看ID为 'sk2131421' 的邮件内容，"
            "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'，"
            "同时做这四件事"
                     )
        ]
    },
    config=config,
)

rprint(response)

### 2.3 举例2:指明工具调用请求决策

In [ ]:
from langgraph.types import Command

weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "广州市", "is_forcast": True}
    }
}

new_decision = {
    "type": "approve"
}

send_email_decision = {
    "type": "approve"
}

decisions = {
    "decisions": []
}

interrupts = response.get("__interrupt__", [])
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(new_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config,
    )

    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

## 3、PIIMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection

API：https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware

敏感信息保护。用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。

### 3.1 参数说明

- pii_type：检测的pii数据类型，可以是内置类型或者自定义类型，内置类型有：
    - email：电子邮箱地址。
    - credit_card：信用卡号。
    - url：网址。
    - mac_address：设备mac地址。
    - ip：IP地址。
- strategy：处理pii信息的策略。支持四种选择：
    - redact：将检测到的PII信息用字符串 [REDACTED_[PII_TYPE]] 替换，其中的 PII_TYPE是上面提到的具体类型，比如 [REDACTED_EMAIL] 、[REDACTED_CREDIT_CARD] 这样的标签。完全 “擦除/隐藏” 真实内容。适合日志清洗、合规需求、公开输出时隐藏敏感内容。
    - mask：用 *** 将PII信息的前面一部分信息遮蔽。比如信用卡号可能变成 ****-****-****-1234 （只保留最后几位/部分可见），邮箱可能保留域名部分 + 隐藏用户名的一部分等 — 既隐藏大部分敏感信息，又保留了一点“可辨识性”（比如账号后四位、域名等），适合用户服务界面 / 前端显示 / 需要部分可识别但不泄露完整敏感内容的场景。
    - hash：用检测到的PII信息的 哈希值 替代原值。比如 <email_hash:a1b2c3d4> 。适合analytics、调试 (debug)、统计分析、匿名追踪等场景。
    - block：如果检测到PII信息， 直接抛出异常 。适合对隐私要求极高、绝不允许泄露任何敏感信息的场景。
- detector：自定义PII检测函数或正则表达式。如果没有提供则使用内置的检测函数。LangChain为每种PII信息定制了专门的检测函数。
```
BUILTIN_DETECTORS: dict[str, Detector] = {
    "email": detect_email,
    "credit_card": detect_credit_card,
    "ip": detect_ip,
    "mac_address": detect_mac_address,
    "url": detect_url,
}
```
- apply_to_input：是否在调用模型前检测，默认为True。
- apply_to_output：是否在调用模型后检测，默认为False。
- apply_to_tool_results：是否在工具调用后检测其输出，默认为False。

通常我们 只在模型调用前 检测。因为PII检测的主要目的是避免将敏感信息发送给模型服务导致信息泄露。

### 3.2 举例1:使用内置检测器

In [ ]:
from langchain.agents.middleware import PIIMiddleware
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

agent = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    帮我向 123456@qq.com 发送一封邮件,
    同时查看银行卡号： 5105-1051-0510-5100 的余额,
    访问 https://localhost:12345,
    确认这是不是 MAC地址： 11-11-11-11-11-11.
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

try:
    agent.invoke({
        "messages": [HumanMessage("看看这个ip能不能ping通：127.0.0.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

### 3.3 自定义检测器/函数

有两种方式创建自定义的检测器：
- 正则表达式：简单的模式匹配；
- 自定义函数：带有验证的复杂检测逻辑。检测器函数必须接受一个字符串（内容）并返回匹配项

In [ ]:
import re
from langchain.agents.middleware import PIIMatch, PIIMiddleware

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

def detect_ssn(content: str) -> list[PIIMatch]:
    """Detect SSNs with validation."""
    matches: list[PIIMatch] = []
    pattern = r"\d{3}-\d{2}-\d{4}"
    for match in re.finditer(pattern, content):
        ssn = match.group(0)
        # Validate: first 3 digits shouldn't be 000, 666, or 900-999
        first_three = int(ssn[:3])
        if first_three not in [0, 666] and not (900 <= first_three <= 999):
            matches.append({
                "type": "ssn",
                "value": ssn,
                "start": match.start(),
                "end": match.end(),
            })
    return matches

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        # Method 1: Regex pattern string
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]+",
            strategy="hash",
        ),
        # Method 2: Custom detector function
        PIIMiddleware(
            "ssn",
            detector=detect_ssn,
            strategy="hash",
        )
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("这是不是有效的 API_KEY： sk-awef23AFEfaafaefa,"
                              "访问 https://localhost:12345, 筛选下面合法的ssn：101-01-0001,202-02-0002,303-03-0003,404-04-0004,505-05-0005,606-06-0006,000-12-3456,666-12-3456,900-12-3456,999-12-3456,123-00-4567,123-45-0000,12-345-678,123-45-67890,123-4A-6789,123.45.6789")]
})

for msg in response["messages"]:
    msg.pretty_print()

## 4、TodoListMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#to-do-list

API：https://reference.langchain.com/python/langchain/agents/middleware/todo/TodoListMiddleware

TodoListMiddleware中间件赋予了Agent 任务规划 和 追踪进度 的能力，可以 应对复杂的多步任务 。

比如，当一个大任务需要被拆解为 3 个以上的子任务，且前面的步骤是后面步骤的前提时，如果不列Todo 列表，大模型在执行到第 3 步时，很容易忘记自己最初的目标，或者在工具返回大量报错信息后“应激”，直接跳过验证去回答用户。

此时， TodoListMiddleware 中间件强制它把计划挂在全局状态里，时刻提醒它“下一步该干什么”。
```
你的任务是否需要拆解？
├── 否 (比如：问答、翻译、单次函数调用) ──> ❌ 绝不需要，浪费算力
└── 是 (比如：写一个包含多文件的工程)
    └── 步骤是否多变且需要应对失败？
        ├── 否 (步骤完全固定，如 A->B->C) ──> ❌ 传统的LangGraph线性节点即可
        └── 是 (AI 需要边做边调计划) ──> 引入 TodoListMiddleware
```
典型场景：
- 任务链路长、步骤多，且有严格的先后依赖关系
- 需要在前端 UI 界面实时展示 Agent 的“思考与执行进度”
- ....

To-do list的创建和维护是通过调用 `write_todos`工具 实现的。

### 4.1 参数说明

- system_prompt：自定义指导todo列表使用的提示词。不提供则使用内置提示词，通常不必提供。
- tool_description：自定义write_tools工具的描述信息。不提供则使用内置描述，通常不必提供。



### 4.2 举例

准备测试用例

In [ ]:
# ../todo_workspace/my_add.py
def add(a: int, b: int) -> int:
    """返回两个整数之和"""
    return a - b

In [ ]:
# ../todo_workspace/test_my_add.py
from my_add import add

def test_add():
    assert add(1, 2) == 3
    assert add(-1, 2) == 0
    assert add(0, 0) == 0
    assert add(10, -5) == 5

In [ ]:
from langchain.agents.middleware import TodoListMiddleware
import subprocess
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from rich import print as rprint
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = '.') -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")

@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"

@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。**无论任务看起来多简单，都必须先调用 write_todos 工具制定待办事项**，"
        "然后再按清单逐步执行。工作全部在工作区下进行。"
    )
)

final_state = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    }
)

rprint(final_state)

为了让 TodoListMiddleware 生效，Agent、工具和中间件三者之间必须满足特定的协同契约：
```
[用户请求] -> "修复 my_add.py"
      │
      ▼
[Agent 思考] -> 意识到是多步骤复杂任务
      │
      ▼
[触发工具] -> 调用 write_todos(tasks=[...])
      │
    ┌─┴────────────────────────┐
    │ TodoListMiddleware 拦截│-> 自动解析工具参数，更新 State 中的 {"todos": [...]}
    └─┬────────────────────────┘
      │
      ▼
[继续执行] -> 读取文件、修改、测试...
      │
      ▼
[最终返回] -> final_state 携带了被中间件更新过的最新 "todos" 列表
```

① todos列表的维护是通过工具调用实现的

此处展示了一次完整的todos列表更新流程
```
[assistant]: 测试确认了问题：`add(2, 3)` 返回了 `-1` 而不是 `5`。现在标记第一个任务为完成，并开始第二个任务：

===== TOOL_CALL =====
tool_name=write_todos
tool_args={'todos': [{'content': '运行现有测试以确认代码问题', 'status':'completed'}, {'content': '修复 my_add.py 中的 add 函数，使其正确执行加法','status': 'in_progress'}, {'content': '再次运行测试以验证修复是否成功', 'status':
'pending'}]}
tool_call_id=call_00_LC5HmpdJ3PAZwkcImJkzQytL

===== TOOL_RESULT =====
tool_call_id=call_00_LC5HmpdJ3PAZwkcImJkzQytL
----- -> 调用结果 <- -----
Updated todo list to [{'content': '运行现有测试以确认代码问题', 'status':'completed'}, {'content': '修复 my_add.py 中的 add 函数，使其正确执行加法','status': 'in_progress'}, {'content': '再次运行测试以验证修复是否成功', 'status':'pending'}]
----- -> 调用结果 <- -----
```
② todos列表的信息分为两部分：status和content，前者是状态，后者是内容

待办事项的状态共有三种取值：
- in_progress：正在进行
- completed：已完成
- pending：待执行

③ 每进行一个步骤，agent会更新To-do lists